In [10]:
#imports
import requests, zipfile, io, os
from pathlib import Path
import pandas as pd



#methods

def get_base_fn(file_path):
    base_name, _ = os.path.splitext(file_path)
    return base_name

def change_extension(file_path, new_extension):
    # base_name, _ = os.path.splitext(file_path)
    base_name = get_base_fn(file_path)
    new_file_path = base_name + "." + new_extension
    os.rename(file_path, new_file_path)
    print(f"{file_path} is now {new_file_path}" )
    

In [11]:
root = Path()


# create folder inside root, called data
# create folder inside root/data called static
#creating data storage folders in root
data_folder = 'data'
data_pth = root / data_folder
data_pth.mkdir(parents=True, exist_ok=True)
static_folder = 'static'
static_pth = data_pth / static_folder
static_pth.mkdir(parents=True, exist_ok=True)


# download gtfs.zip from https://data.foli.fi/gtfs/gtfs.zip and place inside data.
# unzip data/gtfs.zip into static.
zip_url =  r'https://data.foli.fi/gtfs/gtfs.zip'
r = requests.get(zip_url, stream=True)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall(static_pth)







In [12]:
# rename all files to suffix .csv
for f in static_pth.iterdir():
    if f.name.endswith('.txt'):
        change_extension(f, 'csv')


data/static/stops.txt is now data/static/stops.csv
data/static/trip_notes.txt is now data/static/trip_notes.csv
data/static/agency.txt is now data/static/agency.csv
data/static/stop_times.txt is now data/static/stop_times.csv
data/static/calendar_dates.txt is now data/static/calendar_dates.csv
data/static/trips.txt is now data/static/trips.csv
data/static/calendar.txt is now data/static/calendar.csv
data/static/shapes.txt is now data/static/shapes.csv
data/static/routes.txt is now data/static/routes.csv
data/static/feed_info.txt is now data/static/feed_info.csv
data/static/translations.txt is now data/static/translations.csv


In [35]:
#read all 11 .csv files rand transform into pandas datasets.
all_dfs = {}

for f in static_pth.glob('*.csv'):
    all_dfs[f.stem] = pd.read_csv(f)
#create table names called the filename and column names = column names. 


In [ ]:
for df in all_dfs:
    print(df)
    print(all_dfs[df].columns)
    print(all_dfs[df].head())


"""
-- trip_notes.csv
-- columns: trip_id,abbreviation,description,lang

CREATE TABLE IF NOT EXISTS trip_notes (
    trip_id text PRIMARY KEY, 
    abbreviation text NOT NULL, 
    description text, 
    language text
);
-- trips.csv
-- columns: route_id,service_id,trip_id,trip_headsign,direction_id,block_id,shape_id,wheelchair_accessible,bikes_allowed

CREATE TABLE IF NOT EXISTS trips (
    trip_id text NOT NULL PRIMARY KEY,
    route_id INTEGER NOT NULL,
    service_id INTEGER NOT NULL,
    trip_headsign text,
    direction_id INTEGER, --boolean, only 0,1 is integer best for that
    block_id INTEGER,
    shape_id INTEGER,
    wheelchair_accessible INTEGER, --boolean 
    bikes_allowed INTEGER
    FOREIGN KEY (trip_id) REFERENCES trip_notes(trip_id)
);

-- stops.csv
-- columns:stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding,platform_code

CREATE TABLE IF NOT EXISTS stops(
    stop_id INTEGER PRIMARY KEY, 
    stop_code INTEGER NOT NULL,
    stop_name text,
    stop_desc text,
    stop_lat REAL,
    stop_lon REAL,
    zone_id text,
    stop_url text,
    location_type INTEGER,
    parent_station text,
    stop_timezone text,
    wheelchair_boarding INTEGER,
    platform_code text

);

-- stops_translations and trip_headsign_translations was one csv, called translations.csv
-- translations.csv not that easy, contains two types of rows
-- columns: table_name,field_name,language,translation,record_id,field_value
-- rowtype1: stops,stop_name,sv,Satava,386,
-- rowtype2: trips,trip_headsign,sv,Lundo-Tarvasjoki-Koskis,,Lieto-Tarvasjoki-Koski Tl

CREATE TABLE IF NOT EXISTS stops_translations(
    stop_id INTEGER, --corresponds directly to the original finnish name in stops.csv so foreign key to stop_id 
    language text,
    translation text, -- the translation of stop_id in stops.csv
    
);

CREATE TABLE IF NOT EXISTS trip_headsign_translations(
    record_id INTEGER PRIMARY KEY, -- arrived at through searching for field_value inside trips.csv, but they have multiple entries, uncertain which one to use 
    language,
    translation,
    field_value
);

"""



calendar_dates
Index(['service_id', 'date', 'exception_type'], dtype='str')
   service_id      date  exception_type
0  1001010100  20260810               1
1  1001010100  20260811               1
2  1001010100  20260812               1
3  1001010100  20260813               1
4  1001010100  20260814               1
stop_times
Index(['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence',
       'stop_headsign', 'pickup_type', 'drop_off_type', 'shape_dist_traveled',
       'timepoint'],
      dtype='str')
                trip_id arrival_time departure_time  stop_id  stop_sequence  \
0  00026078__1039020165     19:19:00       19:19:00     1644              0   
1  00026078__1039020165     19:19:46       19:19:46      449              1   
2  00026078__1039020165     19:20:40       19:20:40      450              2   
3  00026078__1039020165     19:21:20       19:21:20     1645              3   
4  00026078__1039020165     19:22:05       19:22:05      451              4   


'\nCREATE TABLE IF NOT EXISTS trip_notes (\n    trip_id text PRIMARY KEY, \n    abbreviation text NOT NULL, \n    description text, \n    language text\n);\n\nCREATE TABLE IF NOT EXISTS trips (\n    route_id INTEGER PRIMARY KEY,\n    service_id INTEGER NOT NULL,\n    trip_id text NOT NULL,\n    trip_headsign text,\n    direction_id INTEGER, --boolean, only 0,1 is integer best for that\n    block_id INTEGER,\n    shape_id INTEGER,\n    wheelchair_accessible INTEGER, --boolean \n    bikes_allowed INTEGER\n);\n\nCREATE TABLE IF NOT EXISTS stops(\nstop_id,\nstop_code,\nstop_name,\nstop_desc,\nstop_lat,\nstop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding,platform_code\n\n);\n\nCREATE TABLE IF NOT EXISTS translations(\n    table_name ,\n    field_name,\n    language,\n    translation,\n    record_id,\n    field_value\n\n);\n'